In [ ]:
import pandas as pd
import numpy as np

def load_real_data(filepath):
    df = pd.read_csv(filepath)

    df = df.dropna(how="all", axis=0)
    df = df.dropna(how="all", axis=1)

    df = df.interpolate().fillna(df.mean(numeric_only=True))

    feature_cols = [
        "Rainfall",
        "Temperature",
        "Soil_Moisture",
        "NDVI",
        "Groundwater_Level",
        "Evapotranspiration"
    ]
    target_col = "SVI"

    X = df[feature_cols].to_numpy(dtype=float)
    y = df[target_col].to_numpy(dtype=float)

    return X, y, df

In [ ]:
from sklearn.preprocessing import MinMaxScaler

def preprocess_features(X):
    scaler = MinMaxScaler()
    X_scaled = scaler.fit_transform(X)
    return X_scaled, scaler

In [ ]:
from qiskit.circuit.library import ZZFeatureMap
from qiskit_machine_learning.kernels import FidelityQuantumKernel

def create_quantum_kernel(feature_dim=6, reps=2):
    feature_map = ZZFeatureMap(feature_dimension=feature_dim, reps=reps)
    quantum_kernel = FidelityQuantumKernel(feature_map=feature_map)
    return quantum_kernel, feature_map

In [ ]:
def apply_nystrom_qke(quantum_kernel, X, m=20, seed=42):
    n_samples = len(X)
    m = min(m, n_samples)

    rng = np.random.default_rng(seed)
    landmark_idx = rng.choice(n_samples, size=m, replace=False)
    X_landmarks = X[landmark_idx]

    K_nm = quantum_kernel.evaluate(X, X_landmarks)
    K_mm = quantum_kernel.evaluate(X_landmarks, X_landmarks)

    # Nystrom features: K_nm @ K_mm^(-1/2), so X_qke @ X_qke.T approximates the full kernel
    eigvals, eigvecs = np.linalg.eigh(K_mm)
    eigvals = np.maximum(eigvals, 1e-8)
    K_mm_inv_sqrt = eigvecs @ np.diag(1.0 / np.sqrt(eigvals)) @ eigvecs.T

    X_qke = K_nm @ K_mm_inv_sqrt
    return X_qke

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

def train_qrf_model(X_qke, y, test_size=0.2, random_state=42):
    X_train, X_test, y_train, y_test = train_test_split(
        X_qke, y, test_size=test_size, random_state=random_state
    )

    rf_model = RandomForestRegressor(
        n_estimators=100, random_state=random_state
    )
    rf_model.fit(X_train, y_train)

    y_pred = rf_model.predict(X_test)
    score = r2_score(y_test, y_pred)

    print(f"QRF trained, R^2 = {score:.3f}")

    return rf_model, score, y_pred, y_test

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score, mean_squared_error

def evaluate_qrf_performance(y_true, y_pred, show_plot=True):
    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    print(f"QRF Evaluation:")
    print(f"   R^2 Score: {r2:.3f}")
    print(f"   RMSE: {rmse:.3f}")

    if show_plot:
        plt.figure(figsize=(6, 6))
        plt.scatter(y_true, y_pred, alpha=0.7, edgecolor='k')
        plt.plot([min(y_true), max(y_true)], [min(y_true), max(y_true)], 'r--', label="Perfect Prediction")
        plt.xlabel("True SVI (Observed)")
        plt.ylabel("Predicted SVI (QRF Output)")
        plt.title("Quantum Random Forest Performance")
        plt.legend()
        plt.grid(True)
        plt.show()

    metrics = {"R2": r2, "RMSE": rmse}
    return metrics

In [ ]:
def generate_drought_risk_vector(model, X_all, region_names=None):
    y_pred_all = model.predict(X_all)

    # Lower SVI = more severe drought, so we invert the scale
    risk_raw = -y_pred_all
    risk_norm = (risk_raw - risk_raw.min()) / (risk_raw.max() - risk_raw.min())

    if region_names is None:
        region_names = [f"R{i+1}" for i in range(len(risk_norm))]

    drought_risk = {region_names[i]: float(risk_norm[i]) for i in range(len(region_names))}

    print("Drought risk vector generated:")
    print(pd.Series(drought_risk))

    return drought_risk

In [ ]:
def run_qrf_pipeline(filepath, region_names=None, m=20, bandwidth=0.1):
    X, y, df = load_real_data(filepath)
    print(f"Loaded dataset: {X.shape[0]} samples, {X.shape[1]} features")

    X_scaled, scaler = preprocess_features(X)
    quantum_kernel, feature_map = create_quantum_kernel(feature_dim=X_scaled.shape[1])
    X_qke = apply_nystrom_qke(quantum_kernel, X_scaled*bandwidth, m=m)

    rf_model, score, y_pred, y_test = train_qrf_model(X_qke, y)
    metrics = evaluate_qrf_performance(y_test, y_pred)

    drought_risk = generate_drought_risk_vector(rf_model, X_qke, region_names)

    print("\nPipeline complete")
    return drought_risk, metrics